# 🚢 Notebook 3: Bulkhead Isolation

Ship hulls are split into watertight **bulkheads** so one flooded compartment doesn't sink the boat. In services, we split the worker pool the same way: one slow dependency can't starve every caller of every other dependency.


## 🛠️ Setup

```bash
cd 04-patterns/resilience
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: a single shared thread pool

In [ ]:
import time, threading
from concurrent.futures import ThreadPoolExecutor

def slow_dep():  time.sleep(2.0); return 'slow'
def fast_dep():  time.sleep(0.05); return 'fast'

shared = ThreadPoolExecutor(max_workers=4)

t0 = time.perf_counter()
# 4 slow callers fill the pool
for _ in range(4): shared.submit(slow_dep)
# Now a fast caller arrives — but the pool is full
fut = shared.submit(fast_dep)
print('fast result:', fut.result(), 'after', round(time.perf_counter()-t0,2),'s')
shared.shutdown(wait=True)


Even though the fast call would take 50 ms, it waited for the slow ones — a *latency injection* across unrelated workloads.

## 🟩 BETTER: per-dependency pools

In [ ]:
slow_pool = ThreadPoolExecutor(max_workers=2)
fast_pool = ThreadPoolExecutor(max_workers=2)

t0 = time.perf_counter()
for _ in range(4): slow_pool.submit(slow_dep)
fut = fast_pool.submit(fast_dep)
print('fast result:', fut.result(), 'after', round(time.perf_counter()-t0,2),'s')
slow_pool.shutdown(wait=False); fast_pool.shutdown(wait=False)


Slow calls stay isolated; the fast endpoint is unaffected.

## 🧠 Where to apply

- One pool per **downstream**, or per **tenant**, or per **endpoint criticality**.
- Combine with **timeouts** so a stuck task doesn't sit forever.
- In async code the equivalent is per-dependency `asyncio.Semaphore`.
- Pair with circuit breakers: each bulkhead has its own breaker.
